# Análisis de actividad - Joomla / PostgreSQL
Cuaderno precargado: se conecta a la base de datos `database` y grafica actividad de sesiones.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore', message='Attempting to set identical low and high xlims')
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

# Variables de conexión tomadas del entorno del contenedor (definidas en docker-compose.yml)
DB_HOST = os.environ.get('POSTGRES_HOST', 'database')
DB_NAME = os.environ.get('POSTGRES_DB', 'joomladb')
DB_USER = os.environ.get('POSTGRES_USER', 'joomla_user')
DB_PASS = os.environ.get('POSTGRES_PASSWORD', 'joomla_pass123')

engine = create_engine(f'postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:5432/{DB_NAME}')

In [ ]:
# Consulta de sesiones activas de Joomla (tabla generada al instalar el CMS)
# Nota: prefijo real de tablas confirmado con \dt en psql: 'joom_'
query = 'SELECT time, userid, guest FROM "joom_session"'
df_sesiones = pd.read_sql(query, engine)
df_sesiones['fecha'] = pd.to_datetime(df_sesiones['time'], unit='s')
df_sesiones.head()

In [ ]:
# Gráfico: sesiones agrupadas por hora (tráfico/actividad del portal)
# Si la tabla está vacía (aún no navegaste el sitio como visitante), se avisa en vez de fallar
if df_sesiones.empty:
    print('No hay sesiones registradas todavía. Navega http://localhost/ (frontend) y vuelve a ejecutar esta celda.')
else:
    conteo = df_sesiones.set_index('fecha').resample('1h').size()
    plt.figure(figsize=(10, 4))
    conteo.plot(kind='line', marker='o')
    plt.title('Sesiones de Joomla por hora')
    plt.xlabel('Hora')
    plt.ylabel('Cantidad de sesiones')
    plt.tight_layout()
    plt.show()

In [ ]:
# Gráfico: distribución de sesiones invitado vs autenticado
if df_sesiones.empty:
    print('No hay sesiones registradas todavía. Navega http://localhost/ (frontend) y vuelve a ejecutar esta celda.')
else:
    df_sesiones['tipo'] = df_sesiones['guest'].map({1: 'Invitado', 0: 'Autenticado'})
    df_sesiones['tipo'].value_counts().plot(kind='bar', figsize=(6, 4), title='Sesiones invitado vs autenticado')
    plt.tight_layout()
    plt.show()